In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [18]:
df = pd.read_csv("promoter_seq_TPM.csv")
df = df.dropna()

# ---- Trim to first 1000 bp ----
df["promoter_sequence"] = df["promoter_sequence"].str[:1000]

# ---- Pad sequences shorter than 1000 bp ----
df["promoter_sequence"] = df["promoter_sequence"].apply(
    lambda s: s.ljust(1000, "N")  # pad on right with N
)

# Confirm all sequences are now 1000 bp
print("Unique sequence lengths:", df["promoter_sequence"].apply(len).unique())


Unique sequence lengths: [1000]


In [19]:
BASE2IDX = {"A":0, "C":1, "G":2, "T":3}

def one_hot_encode(seq):
    arr = np.zeros((4, len(seq)), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in BASE2IDX:
            arr[BASE2IDX[b], i] = 1.0
    return arr


In [20]:
class PromoterDataset(Dataset):
    def __init__(self, seqs, exprs):
        self.seqs = seqs
        self.exprs = torch.tensor(exprs, dtype=torch.float32)

        # Normalize expression
        self.mean = self.exprs.mean()
        self.std = self.exprs.std() + 1e-8
        self.exprs_norm = (self.exprs - self.mean) / self.std

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = one_hot_encode(self.seqs[idx])  # shape (4, 1000)
        x = torch.tensor(x, dtype=torch.float32)
        y = self.exprs_norm[idx]
        return x, y


In [26]:
class PromoterCNN(nn.Module):
    def __init__(self, seq_len):
        super().__init__()

        self.conv1 = nn.Conv1d(4, 32, kernel_size=8, padding=4)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=8, padding=4)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=8, padding=4)
        self.bn3 = nn.BatchNorm1d(128)

        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.5)

        # Dynamically compute flatten size
        with torch.no_grad():
            dummy = torch.zeros(1, 4, seq_len)
            x = self.pool(F.relu(self.bn1(self.conv1(dummy))))
            x = self.dropout(x)
            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = self.dropout(x)
            x = self.pool(F.relu(self.bn3(self.conv3(x))))
            x = self.dropout(x)
            self.flat_dim = x.numel()

        # Smaller FC layers (prevents overfitting)
        self.fc1 = nn.Linear(self.flat_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x).squeeze(-1)


In [22]:
sequences = df["promoter_sequence"].tolist()
expressions = df["mean_TPM"].astype(float).tolist()

train_seqs, test_seqs, train_expr, test_expr = train_test_split(
    sequences, expressions, test_size=0.1, random_state=42
)

train_seqs, val_seqs, train_expr, val_expr = train_test_split(
    train_seqs, train_expr, test_size=0.1, random_state=42
)


In [27]:
def train_model(train_ds, val_ds, lr=1e-3, batch_size=64, epochs=30):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = PromoterCNN(seq_len=1000).to(device)

    # weight decay helps generalization!
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience = 5
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

        train_loss = total_loss / len(train_ds)

        # ------------------
        # Validation
        # ------------------
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = loss_fn(pred, y)
                val_loss += loss.item() * x.size(0)

        val_loss /= len(val_ds)

        print(f"Epoch {epoch:03d} | train={train_loss:.4f} | val={val_loss:.4f}")

        # Early stopping
        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model


In [28]:
train_ds = PromoterDataset(train_seqs, train_expr)
val_ds   = PromoterDataset(val_seqs, val_expr)
test_ds  = PromoterDataset(test_seqs, test_expr)

model = train_model(train_ds, val_ds, epochs=30, batch_size=64)


Using device: cuda
Epoch 001 | train=1.0892 | val=1.0002
Epoch 002 | train=1.0003 | val=0.9995
Epoch 003 | train=1.0000 | val=0.9994
Epoch 004 | train=1.0000 | val=0.9995
Epoch 005 | train=1.0000 | val=0.9994
Epoch 006 | train=1.0000 | val=0.9995
Epoch 007 | train=1.0001 | val=0.9995
Epoch 008 | train=1.0001 | val=0.9994
Epoch 009 | train=1.0000 | val=0.9995
Epoch 010 | train=1.0001 | val=0.9994
Epoch 011 | train=1.0000 | val=0.9995
Epoch 012 | train=1.0001 | val=0.9994
Epoch 013 | train=1.0000 | val=0.9994


KeyboardInterrupt: 

In [24]:
torch.save({
    "model_state": model.state_dict(),
    "mean": train_ds.mean.item(),
    "std": train_ds.std.item(),
    "seq_len": 1000,
}, "promoter_cnn_tpm.pt")

print("Model saved as promoter_cnn_tpm.pt")


NameError: name 'model' is not defined

### Transformer

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


In [51]:
# ============================================================
# 1. IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import time

# ============================================================
# 2. LOAD DATASET
# ============================================================
df = pd.read_csv("promoter_seq_TPM.csv").dropna(subset=["promoter_sequence", "mean_TPM"])

# Trim or pad sequences to exactly 1000 bp
df["promoter_sequence"] = df["promoter_sequence"].str[:1000]
df["promoter_sequence"] = df["promoter_sequence"].apply(lambda s: s.ljust(1000, "N"))

sequences = df["promoter_sequence"].tolist()
expressions = df["mean_TPM"].astype(float).tolist()

print("Dataset size:", len(sequences))


# ============================================================
# 3. ONE-HOT ENCODING + DATASET CLASS
# ============================================================
BASE2IDX = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot_encode(seq):
    seq = seq.upper()
    L = len(seq)
    arr = np.zeros((4, L), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in BASE2IDX:
            arr[BASE2IDX[b], i] = 1.0
    return arr

class PromoterDataset(Dataset):
    def __init__(self, seqs, exprs):
        self.seqs = seqs
        self.exprs = torch.tensor(exprs, dtype=torch.float32)

        # Normalize labels
        self.mean = self.exprs.mean()
        self.std = self.exprs.std() + 1e-8
        self.exprs_norm = (self.exprs - self.mean) / self.std

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = torch.tensor(one_hot_encode(self.seqs[idx]), dtype=torch.float32)  # (4,1000)
        y = self.exprs_norm[idx]
        return x, y


# ============================================================
# 4. SPLIT INTO TRAIN / VAL / TEST
# ============================================================
train_seqs, test_seqs, train_expr, test_expr = train_test_split(
    sequences, expressions, test_size=0.1, random_state=42
)

train_seqs, val_seqs, train_expr, val_expr = train_test_split(
    train_seqs, train_expr, test_size=0.1, random_state=42
)

train_ds = PromoterDataset(train_seqs, train_expr)
val_ds   = PromoterDataset(val_seqs, val_expr)
test_ds  = PromoterDataset(test_seqs, test_expr)


# ============================================================
# 5. DILATED CNN MODEL
# ============================================================
class DilatedResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) // 2 * dilation

        self.conv1 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        x = self.dropout(x)
        x = F.relu(x + residual)
        return x


class PromoterDilatedCNN(nn.Module):
    def __init__(self, seq_len=1000, base_channels=64, num_blocks=6,
                 kernel_size=7, dropout=0.2):
        super().__init__()

        self.conv_in = nn.Conv1d(4, base_channels, kernel_size=kernel_size,
                                 padding=(kernel_size - 1) // 2)
        self.bn_in = nn.BatchNorm1d(base_channels)

        blocks = []
        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(
                DilatedResidualBlock(
                    channels=base_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                )
            )
        self.blocks = nn.Sequential(*blocks)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(base_channels, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))
        x = self.blocks(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)


# ============================================================
# 6. TRAINING FUNCTION WITH ETA
# ============================================================
def train_dilated_cnn(train_ds, val_ds, lr=1e-3, batch_size=64, epochs=40):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = PromoterDilatedCNN(seq_len=1000).to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience = 6
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        # ===== TRAIN =====
        model.train()
        total_loss = 0.0
        num_batches = len(train_loader)

        for batch_i, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

            if batch_i % max(1, num_batches // 10) == 0:
                pct = batch_i / num_batches * 100
                print(f"  Epoch {epoch}/{epochs}: {pct:.1f}% complete", end="\r")

        train_loss = total_loss / len(train_ds)

        # ===== VAL =====
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += loss_fn(model(x), y).item() * x.size(0)

        val_loss /= len(val_ds)

        epoch_time = time.time() - epoch_start
        eta_min = (epochs - epoch) * epoch_time / 60

        print(f"\nEpoch {epoch:03d} | train MSE={train_loss:.4f} | val MSE={val_loss:.4f} "
              f"| epoch time: {epoch_time:.1f}s | ETA: {eta_min:.1f} min")

        # Early stopping
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state)
    return model


# ============================================================
# 7. TRAIN THE MODEL
# ============================================================
model = train_dilated_cnn(train_ds, val_ds, epochs=40, batch_size=64)


# ============================================================
# 8. EVALUATE ON TEST SET
# ============================================================
# ============================================================
# Correct evaluation on test set
# ============================================================

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

test_loader = DataLoader(test_ds, batch_size=64)
preds = []
true_vals = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        pred = model(x).cpu().numpy()
        preds.append(pred)
        true_vals.append(y.numpy())

preds = np.concatenate(preds)
true_vals = np.concatenate(true_vals)

# Get normalization stats
mean = train_ds.mean.item()
std = train_ds.std.item()

# Unnormalize BOTH prediction & ground truth
preds = preds * std + mean
true_vals = true_vals * std + mean    # <-- THIS FIXES THE 23 MSE PROBLEM

mse_test = np.mean((preds - true_vals)**2)
print("\nCorrected TEST MSE:", mse_test)



# ============================================================
# 9. FUNCTION FOR WGAN SEQUENCE EVALUATION
# ============================================================
def evaluate_sequences(model, seq_list, mean, std, batch_size=64):
    """
    seq_list: list of promoter sequences (strings)
    Returns predicted TPM values.
    """

    # Trim & pad
    seq_list = [s[:1000].ljust(1000, "N") for s in seq_list]

    device = next(model.parameters()).device
    model.eval()

    preds = []
    with torch.no_grad():
        for i in range(0, len(seq_list), batch_size):
            batch = seq_list[i:i+batch_size]
            x = np.stack([one_hot_encode(s) for s in batch])
            x = torch.tensor(x, dtype=torch.float32).to(device)
            pred = model(x).cpu().numpy()
            preds.append(pred)

    preds = np.concatenate(preds)
    # unnormalize
    return preds * std + mean


print("\nReady to evaluate WGAN sequences using evaluate_sequences(...)")


Dataset size: 19942
Using device: cuda
  Epoch 1/40: 98.8% complete
Epoch 001 | train MSE=0.9861 | val MSE=1.0183 | epoch time: 20.8s | ETA: 13.5 min
  Epoch 2/40: 98.8% complete
Epoch 002 | train MSE=0.9560 | val MSE=0.9575 | epoch time: 20.8s | ETA: 13.2 min
  Epoch 3/40: 98.8% complete
Epoch 003 | train MSE=0.9472 | val MSE=0.9484 | epoch time: 21.4s | ETA: 13.2 min
  Epoch 4/40: 98.8% complete
Epoch 004 | train MSE=0.9406 | val MSE=0.9347 | epoch time: 21.4s | ETA: 12.9 min
  Epoch 5/40: 98.8% complete
Epoch 005 | train MSE=0.9349 | val MSE=0.9501 | epoch time: 22.9s | ETA: 13.3 min
  Epoch 6/40: 98.8% complete
Epoch 006 | train MSE=0.9308 | val MSE=0.9459 | epoch time: 21.5s | ETA: 12.2 min
  Epoch 7/40: 98.8% complete
Epoch 007 | train MSE=0.9253 | val MSE=0.9624 | epoch time: 21.9s | ETA: 12.1 min
  Epoch 8/40: 98.8% complete
Epoch 008 | train MSE=0.9181 | val MSE=0.9725 | epoch time: 21.8s | ETA: 11.6 min
  Epoch 9/40: 98.8% complete
Epoch 009 | train MSE=0.9126 | val MSE=1.031

In [52]:
import numpy as np

print("Train mean:", np.mean(train_expr))
print("Val mean:", np.mean(val_expr))
print("Test mean:", np.mean(test_expr))


Train mean: 3.3380671442058936
Val mean: 3.276594957986438
Test mean: 3.3297509489875594


In [53]:
# ============================================================
# 0. IMPORTS
# ============================================================
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split

# ============================================================
# 1. LOAD DATASET & PREPROCESS
# ============================================================
df = pd.read_csv("promoter_seq_TPM.csv").dropna(subset=["promoter_sequence", "mean_TPM"])

# Trim/pad to exactly 1000 bp
df["promoter_sequence"] = df["promoter_sequence"].str[:1000]
df["promoter_sequence"] = df["promoter_sequence"].apply(lambda s: s.ljust(1000, "N"))

# Raw TPM and log(TPM+1) targets
sequences = df["promoter_sequence"].tolist()
tpm = df["mean_TPM"].astype(float).values
log_tpm = np.log1p(tpm)  # log(TPM+1)

print("Dataset size:", len(sequences))
print("TPM stats:", df["mean_TPM"].describe())
print("log(TPM+1) stats:", pd.Series(log_tpm).describe())

# ============================================================
# 2. ONE-HOT ENCODING + DATASET CLASS (with shared normalization)
# ============================================================
BASE2IDX = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot_encode(seq: str) -> np.ndarray:
    seq = seq.upper()
    L = len(seq)
    arr = np.zeros((4, L), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in BASE2IDX:
            arr[BASE2IDX[b], i] = 1.0
    return arr

class PromoterDataset(Dataset):
    """
    seqs: list of strings
    log_exprs: list/array of log(TPM+1)
    mean, std: normalization stats in log space (shared across splits)
    """
    def __init__(self, seqs, log_exprs, mean=None, std=None):
        self.seqs = seqs
        self.log_exprs = torch.tensor(log_exprs, dtype=torch.float32)

        if mean is None or std is None:
            self.mean = self.log_exprs.mean()
            self.std = self.log_exprs.std() + 1e-8
        else:
            self.mean = torch.tensor(mean, dtype=torch.float32)
            self.std = torch.tensor(std, dtype=torch.float32)

        self.targets = (self.log_exprs - self.mean) / self.std  # normalized log(TPM+1)

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = torch.tensor(one_hot_encode(self.seqs[idx]), dtype=torch.float32)  # (4, L)
        y = self.targets[idx]  # normalized log(TPM+1)
        return x, y

# ============================================================
# 3. TRAIN / VAL / TEST SPLIT (keeping log & raw TPM aligned)
# ============================================================
(
    train_seqs, test_seqs,
    train_log, test_log,
    train_tpm, test_tpm
) = train_test_split(
    sequences, log_tpm, tpm,
    test_size=0.1, random_state=42
)

(
    train_seqs, val_seqs,
    train_log, val_log,
    train_tpm, val_tpm
) = train_test_split(
    train_seqs, train_log, train_tpm,
    test_size=0.1, random_state=42
)

# Build train dataset first to get mean/std
train_ds = PromoterDataset(train_seqs, train_log)
log_mean = train_ds.mean.item()
log_std = train_ds.std.item()

# Rebuild val/test with shared normalization stats
val_ds  = PromoterDataset(val_seqs,  val_log,  mean=log_mean, std=log_std)
test_ds = PromoterDataset(test_seqs, test_log, mean=log_mean, std=log_std)

print("\nTrain/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))
print("Train log(TPM+1) mean/std (used for all):", log_mean, log_std)

# ============================================================
# 4. DILATED RESIDUAL CNN
# ============================================================
class DilatedResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) // 2 * dilation

        self.conv1 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(channels, channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        x = self.dropout(x)
        x = F.relu(x + residual)
        return x


class PromoterDilatedCNN(nn.Module):
    def __init__(self, seq_len=1000, base_channels=64,
                 num_blocks=6, kernel_size=7, dropout=0.2):
        super().__init__()

        self.conv_in = nn.Conv1d(4, base_channels, kernel_size=kernel_size,
                                 padding=(kernel_size - 1) // 2)
        self.bn_in = nn.BatchNorm1d(base_channels)

        blocks = []
        for i in range(num_blocks):
            dilation = 2 ** i  # 1,2,4,8,16,32
            blocks.append(
                DilatedResidualBlock(
                    channels=base_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                )
            )
        self.blocks = nn.Sequential(*blocks)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(base_channels, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, 4, L)
        x = F.relu(self.bn_in(self.conv_in(x)))
        x = self.blocks(x)
        x = self.global_pool(x).squeeze(-1)  # (B, C)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)       # (B,) normalized log(TPM+1)

# ============================================================
# 5. TRAINING FUNCTION (MSE in normalized log space)
# ============================================================
def train_dilated_cnn(train_ds, val_ds, lr=1e-3, batch_size=64, epochs=40):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("\nUsing device:", device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)

    model = PromoterDilatedCNN(seq_len=1000).to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience = 6
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        # ----- TRAIN -----
        model.train()
        total_loss = 0.0
        num_batches = len(train_loader)

        for batch_i, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

            if batch_i % max(1, num_batches // 10) == 0:
                pct = batch_i / num_batches * 100
                print(f"  Epoch {epoch}/{epochs}: {pct:.1f}% complete", end="\r")

        train_loss = total_loss / len(train_ds)

        # ----- VALIDATE -----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += loss_fn(pred, y).item() * x.size(0)

        val_loss /= len(val_ds)
        epoch_time = time.time() - epoch_start
        eta_min = (epochs - epoch) * epoch_time / 60

        print(f"\nEpoch {epoch:03d} | train MSE (norm log)={train_loss:.4f} "
              f"| val MSE (norm log)={val_loss:.4f} "
              f"| epoch time: {epoch_time:.1f}s | ETA: {eta_min:.1f} min")

        # Early stopping
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state)
    return model

# ============================================================
# 6. TRAIN MODEL
# ============================================================
model = train_dilated_cnn(train_ds, val_ds, epochs=40, batch_size=64)

# ============================================================
# 7. EVALUATE ON TEST SET
#    (both in log-space and real TPM space)
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval().to(device)

test_loader = DataLoader(test_ds, batch_size=64)
all_pred_norm = []
all_true_norm = []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)            # normalized log(TPM+1)
        all_pred_norm.append(pred.cpu().numpy())
        all_true_norm.append(y.cpu().numpy())

all_pred_norm = np.concatenate(all_pred_norm)  # normalized log
all_true_norm = np.concatenate(all_true_norm)  # normalized log

# Shared normalization stats (from train)
log_mean = train_ds.mean.item()
log_std  = train_ds.std.item()

# Denormalize to log(TPM+1)
pred_log = all_pred_norm * log_std + log_mean
true_log = all_true_norm * log_std + log_mean

# Back-transform to TPM
pred_tpm = np.expm1(pred_log)
true_tpm = np.expm1(true_log)

# MSE in log space and real TPM space
mse_log = np.mean((pred_log - true_log) ** 2)
mse_tpm = np.mean((pred_tpm - true_tpm) ** 2)

print("\n=== TEST PERFORMANCE ===")
print("MSE in log(TPM+1) space:", mse_log)
print("MSE in TPM space:", mse_tpm)

# ============================================================
# 8. HELPER: EVALUATE ARBITRARY SEQUENCES (e.g., WGAN OUTPUTS)
# ============================================================
def evaluate_sequences_dilated_cnn(model, seq_list, log_mean, log_std, batch_size=64):
    """
    seq_list: list of DNA strings (A/C/G/T/N...), variable length
    Returns: predicted TPM (numpy array)
    """
    # Trim + pad to 1000 bp like training
    seq_list = [s[:1000].ljust(1000, "N") for s in seq_list]

    device = next(model.parameters()).device
    model.eval()

    preds_norm = []
    with torch.no_grad():
        for i in range(0, len(seq_list), batch_size):
            batch = seq_list[i:i+batch_size]
            x = np.stack([one_hot_encode(s) for s in batch])
            x = torch.tensor(x, dtype=torch.float32).to(device)
            pred = model(x)  # normalized log(TPM+1)
            preds_norm.append(pred.cpu().numpy())

    preds_norm = np.concatenate(preds_norm)
    pred_log = preds_norm * log_std + log_mean       # log(TPM+1)
    pred_tpm = np.expm1(pred_log)                    # TPM
    return pred_tpm

print("\nModel + helper ready. Use evaluate_sequences_dilated_cnn(...) for WGAN sequences.")


Dataset size: 19942
TPM stats: count    19942.000000
mean         3.331702
std          1.991156
min          0.708987
25%          1.517486
50%          3.160364
75%          4.736323
max         14.326857
Name: mean_TPM, dtype: float64
log(TPM+1) stats: count    19942.000000
mean         1.356725
std          0.475882
min          0.535901
25%          0.923261
50%          1.425603
75%          1.746818
max          2.729607
dtype: float64

Train/Val/Test sizes: 16152 1795 1995
Train log(TPM+1) mean/std (used for all): 1.3578641414642334 0.4766986072063446

Using device: cuda
  Epoch 1/40: 98.8% complete
Epoch 001 | train MSE (norm log)=0.9825 | val MSE (norm log)=0.9725 | epoch time: 20.7s | ETA: 13.5 min
  Epoch 2/40: 98.8% complete
Epoch 002 | train MSE (norm log)=0.9507 | val MSE (norm log)=0.9836 | epoch time: 19.9s | ETA: 12.6 min
  Epoch 3/40: 98.8% complete
Epoch 003 | train MSE (norm log)=0.9364 | val MSE (norm log)=0.9883 | epoch time: 20.5s | ETA: 12.7 min
  Epoch 4/40: 9

In [54]:
# ============================================================
# 0. IMPORTS
# ============================================================
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split


# ============================================================
# 1. LOAD DATASET
# ============================================================
df = pd.read_csv("promoter_seq_TPM.csv").dropna(subset=["promoter_sequence", "mean_TPM"])

# Trim / pad to 1000 bp
df["promoter_sequence"] = df["promoter_sequence"].str[:1000]
df["promoter_sequence"] = df["promoter_sequence"].apply(lambda s: s.ljust(1000, "N"))

sequences = df["promoter_sequence"].tolist()
tpm = df["mean_TPM"].astype(float).values
log_tpm = np.log1p(tpm)

print("Dataset size:", len(sequences))


# ============================================================
# 2. ONE-HOT ENCODING + DATASET
# ============================================================
BASE2IDX = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot_encode(seq):
    arr = np.zeros((4, len(seq)), dtype=np.float32)
    for i, b in enumerate(seq.upper()):
        if b in BASE2IDX:
            arr[BASE2IDX[b], i] = 1.0
    return arr

class PromoterDataset(Dataset):
    def __init__(self, seqs, log_exprs, mean=None, std=None):
        self.seqs = seqs
        self.log_exprs = torch.tensor(log_exprs, dtype=torch.float32)

        if mean is None or std is None:
            self.mean = self.log_exprs.mean()
            self.std = self.log_exprs.std() + 1e-8
        else:
            self.mean = torch.tensor(mean)
            self.std = torch.tensor(std)

        self.targets = (self.log_exprs - self.mean) / self.std

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        x = torch.tensor(one_hot_encode(self.seqs[idx]), dtype=torch.float32)
        y = self.targets[idx]
        return x, y


# ============================================================
# 3. TRAIN/VAL/TEST SPLIT
# ============================================================
(train_seqs, test_seqs,
 train_log, test_log,
 train_tpm, test_tpm) = train_test_split(
    sequences, log_tpm, tpm, test_size=0.1, random_state=42
)

(train_seqs, val_seqs,
 train_log, val_log,
 train_tpm, val_tpm) = train_test_split(
    train_seqs, train_log, train_tpm, test_size=0.1, random_state=42
)

train_ds = PromoterDataset(train_seqs, train_log)
log_mean = train_ds.mean.item()
log_std  = train_ds.std.item()

val_ds  = PromoterDataset(val_seqs,  val_log,  log_mean, log_std)
test_ds = PromoterDataset(test_seqs, test_log, log_mean, log_std)

print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds))


# ============================================================
# 4. BASENJI-STYLE BLOCKS
# ============================================================

class MultiScaleConv(nn.Module):
    """Conv kernels: 3, 7, 15 (parallel)."""
    def __init__(self, C):
        super().__init__()
        self.conv3  = nn.Conv1d(C, C, kernel_size=3,  padding=1)
        self.conv7  = nn.Conv1d(C, C, kernel_size=7,  padding=3)
        self.conv15 = nn.Conv1d(C, C, kernel_size=15, padding=7)
        self.bn = nn.BatchNorm1d(C)

    def forward(self, x):
        out = self.conv3(x) + self.conv7(x) + self.conv15(x)
        out = self.bn(out)
        return F.relu(out)


class DilatedResidualBlock(nn.Module):
    def __init__(self, C, dilation, dropout=0.15):
        super().__init__()
        self.conv = nn.Conv1d(C, C, kernel_size=7,
                              padding=(7//2)*dilation,
                              dilation=dilation)
        self.bn = nn.BatchNorm1d(C)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = self.dropout(F.relu(self.bn(self.conv(x))))
        return F.relu(out + x)


class BasenjiDilatedCNN(nn.Module):
    def __init__(self, seq_len=1000, C=128, num_blocks=12):
        super().__init__()

        # Multi-scale initial layer
        self.conv_in = MultiScaleConv(4)
        self.proj = nn.Conv1d(4, C, kernel_size=1)

        blocks = []
        for i in range(num_blocks):
            blocks.append(DilatedResidualBlock(C, dilation=2**i))
        self.blocks = nn.Sequential(*blocks)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(C, 128)
        self.fc2 = nn.Linear(128, 1)

        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.conv_in(x)     # multi-scale conv
        x = self.proj(x)        # project to C=128
        x = self.blocks(x)      # deep dilated stack
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)  # normalized log(TPM+1)


# ============================================================
# 5. TRAINING FUNCTION (Huber Loss)
# ============================================================
def train_basenji(train_ds, val_ds, epochs=40, batch_size=32, lr=3e-4):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("\nUsing device:", device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size)

    model = BasenjiDilatedCNN().to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.SmoothL1Loss(beta=0.5)  # Huber

    best_val = float("inf")
    best_state = None
    patience, patience_ctr = 8, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # ----- TRAIN -----
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_ds)

        # ----- VALIDATION -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = loss_fn(pred, y)
                val_loss += loss.item() * x.size(0)
        val_loss /= len(val_ds)

        t1 = time.time()
        print(f"Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} "
              f"| time={t1-t0:.1f}s")

        # Early stopping
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = model.state_dict()
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model


# ============================================================
# 6. TRAIN MODEL
# ============================================================
model = train_basenji(train_ds, val_ds, epochs=40, batch_size=32)


# ============================================================
# 7. EVALUATION IN LOG AND TPM SPACE
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval().to(device)

test_loader = DataLoader(test_ds, batch_size=64)
pred_norm, true_norm = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred_norm.append(model(x).cpu().numpy())
        true_norm.append(y.cpu().numpy())

pred_norm = np.concatenate(pred_norm)
true_norm = np.concatenate(true_norm)

# Denormalize
pred_log = pred_norm * log_std + log_mean
true_log = true_norm * log_std + log_mean

# Convert to TPM
pred_tpm = np.expm1(pred_log)
true_tpm = np.expm1(true_log)

mse_log = np.mean((pred_log - true_log)**2)
mse_tpm = np.mean((pred_tpm - true_tpm)**2)

print("\n=== TEST PERFORMANCE ===")
print("MSE (log TPM+1):", mse_log)
print("MSE (TPM):", mse_tpm)


# ============================================================
# 8. WGAN SCORING HELPER
# ============================================================
def evaluate_sequences_basenji(model, seq_list, log_mean, log_std, batch_size=64):
    seq_list = [s[:1000].ljust(1000, "N") for s in seq_list]
    device = next(model.parameters()).device
    model.eval()

    preds = []
    with torch.no_grad():
        for i in range(0, len(seq_list), batch_size):
            batch = seq_list[i:i+batch_size]
            x = np.stack([one_hot_encode(s) for s in batch])
            x = torch.tensor(x, dtype=torch.float32).to(device)
            pred = model(x)  # normalized log
            preds.append(pred.cpu().numpy())

    preds = np.concatenate(preds)
    pred_log = preds * log_std + log_mean
    pred_tpm = np.expm1(pred_log)
    return pred_tpm

print("\nBasenji-style promoter model ready. Use evaluate_sequences_basenji(...) for WGAN sequences.")


Dataset size: 19942
Train/Val/Test: 16152 1795 1995

Using device: cuda
Epoch 01 | train=0.6434 | val=0.6042 | time=51.9s
Epoch 02 | train=0.6076 | val=0.6293 | time=58.6s
Epoch 03 | train=0.6018 | val=0.6481 | time=58.7s
Epoch 04 | train=0.5972 | val=0.6031 | time=65.6s
Epoch 05 | train=0.5915 | val=0.6117 | time=57.0s
Epoch 06 | train=0.5866 | val=0.6147 | time=51.6s
Epoch 07 | train=0.5791 | val=0.6402 | time=52.1s
Epoch 08 | train=0.5699 | val=0.7035 | time=52.5s
Epoch 09 | train=0.5683 | val=0.6408 | time=52.3s
Epoch 10 | train=0.5564 | val=0.6624 | time=51.4s
Epoch 11 | train=0.5451 | val=0.6582 | time=52.3s
Epoch 12 | train=0.5342 | val=0.7484 | time=52.3s
Early stopping.

=== TEST PERFORMANCE ===
MSE (log TPM+1): 0.3139213
MSE (TPM): 4.9123015

Basenji-style promoter model ready. Use evaluate_sequences_basenji(...) for WGAN sequences.


In [56]:
# ============================================================
# 0. IMPORTS
# ============================================================
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from sklearn.model_selection import train_test_split


# ============================================================
# 1. RC FUNCTION
# ============================================================
COMP = str.maketrans("ACGTNacgtn", "TGCANtgcan")

def reverse_complement(seq: str) -> str:
    return seq.translate(COMP)[::-1]


# ============================================================
# 2. LOAD DATASET
# ============================================================
df = pd.read_csv("promoter_seq_TPM.csv").dropna(subset=["promoter_sequence", "mean_TPM"])

df["promoter_sequence"] = df["promoter_sequence"].str[:1000]
df["promoter_sequence"] = df["promoter_sequence"].apply(lambda s: s.ljust(1000, "N"))

sequences = df["promoter_sequence"].tolist()
tpm = df["mean_TPM"].astype(float).values
log_tpm = np.log1p(tpm)

print("Dataset size:", len(sequences))


# ============================================================
# 3. ONE-HOT ENCODING
# ============================================================
BASE2IDX = {"A":0, "C":1, "G":2, "T":3}

def one_hot_encode(seq):
    arr = np.zeros((4, len(seq)), dtype=np.float32)
    for i, b in enumerate(seq.upper()):
        if b in BASE2IDX:
            arr[BASE2IDX[b], i] = 1.0
    return arr


# ============================================================
# 4. DATASET WITH RC AUGMENTATION
# ============================================================
class PromoterDataset(Dataset):
    def __init__(self, seqs, log_exprs, mean=None, std=None, augment=False):
        self.seqs = seqs
        self.log_exprs = torch.tensor(log_exprs, dtype=torch.float32)
        self.augment = augment   # <-- apply RC augmentation only in training

        if mean is None:
            self.mean = self.log_exprs.mean()
            self.std  = self.log_exprs.std() + 1e-8
        else:
            self.mean = torch.tensor(mean)
            self.std  = torch.tensor(std)

        self.targets = (self.log_exprs - self.mean) / self.std

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]

        # ----- RC augmentation -----
        if self.augment and np.random.rand() < 0.5:
            seq = reverse_complement(seq)

        x = torch.tensor(one_hot_encode(seq), dtype=torch.float32)
        y = self.targets[idx]
        return x, y


# ============================================================
# 5. TRAIN / VAL / TEST SPLIT
# ============================================================
(train_seqs, test_seqs,
 train_log, test_log,
 train_tpm, test_tpm) = train_test_split(
    sequences, log_tpm, tpm, test_size=0.1, random_state=42
)

(train_seqs, val_seqs,
 train_log, val_log,
 train_tpm, val_tpm) = train_test_split(
    train_seqs, train_log, train_tpm, test_size=0.1, random_state=42
)

train_ds = PromoterDataset(train_seqs, train_log, augment=True)
log_mean = train_ds.mean.item()
log_std  = train_ds.std.item()

val_ds  = PromoterDataset(val_seqs,  val_log,  log_mean, log_std, augment=False)
test_ds = PromoterDataset(test_seqs, test_log, log_mean, log_std, augment=False)

print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds))


# ============================================================
# 6. BASENJI-STYLE MODEL
# ============================================================
class MultiScaleConv(nn.Module):
    def __init__(self, C_in, C_out):
        super().__init__()
        self.c3  = nn.Conv1d(C_in, C_out, 3, padding=1)
        self.c7  = nn.Conv1d(C_in, C_out, 7, padding=3)
        self.c15 = nn.Conv1d(C_in, C_out, 15, padding=7)
        self.bn = nn.BatchNorm1d(C_out)

    def forward(self, x):
        out = self.c3(x) + self.c7(x) + self.c15(x)
        return F.relu(self.bn(out))


class DilatedResidualBlock(nn.Module):
    def __init__(self, C, dilation):
        super().__init__()
        self.conv = nn.Conv1d(C, C, 7,
                              padding=(7//2)*dilation,
                              dilation=dilation)
        self.bn = nn.BatchNorm1d(C)
        self.drop = nn.Dropout(0.15)

    def forward(self, x):
        out = F.relu(self.bn(self.conv(x)))
        out = self.drop(out)
        return F.relu(out + x)


class BasenjiDilatedCNN(nn.Module):
    def __init__(self, C=128, num_blocks=12):
        super().__init__()
        self.ms = MultiScaleConv(4, C)

        blocks = []
        for i in range(num_blocks):
            blocks.append(DilatedResidualBlock(C, dilation=2**i))
        self.blocks = nn.Sequential(*blocks)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(C, 128)
        self.fc2 = nn.Linear(128, 1)
        self.drop = nn.Dropout(0.25)

    def forward(self, x):
        x = self.ms(x)
        x = self.blocks(x)
        x = self.pool(x).squeeze(-1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)   # normalized log(TPM+1)


# ============================================================
# 7. TRAINING LOOP (Huber loss)
# ============================================================
def train_model(train_ds, val_ds, epochs=7, batch_size=32, lr=3e-4):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("\nUsing device:", device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size)

    model = BasenjiDilatedCNN().to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.SmoothL1Loss(beta=0.5)

    best_val, patience, wait = float("inf"), 8, 0
    best_state = None

    for epoch in range(1, epochs+1):
        t0 = time.time()

        # TRAIN
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
        train_loss /= len(train_ds)

        # VALIDATION
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += loss_fn(pred, y).item() * x.size(0)
        val_loss /= len(val_ds)

        print(f"Epoch {epoch:02d} | train={train_loss:.4f} | val={val_loss:.4f} | time={time.time()-t0:.1f}s")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return model


model = train_model(train_ds, val_ds, epochs=40, batch_size=32)


# ============================================================
# 8. RC PREDICTION AVERAGING FOR TEST SET
# ============================================================
def predict_with_rc(model, seqs, log_mean, log_std, batch_size=64):
    preds = []
    device = next(model.parameters()).device

    with torch.no_grad():
        for i in range(0, len(seqs), batch_size):
            batch = seqs[i:i+batch_size]
            
            # Original orientation
            x1 = np.stack([one_hot_encode(s) for s in batch])
            x1 = torch.tensor(x1, dtype=torch.float32).to(device)
            
            # Reverse complement
            batch_rc = [reverse_complement(s) for s in batch]
            x2 = np.stack([one_hot_encode(s) for s in batch_rc])
            x2 = torch.tensor(x2, dtype=torch.float32).to(device)

            # Average predictions
            p1 = model(x1)
            p2 = model(x2)
            p = (p1 + p2) / 2.0

            preds.append(p.cpu().numpy())

    preds = np.concatenate(preds)
    log_pred = preds * log_std + log_mean
    tpm_pred = np.expm1(log_pred)
    return log_pred, tpm_pred


# TEST EVALUATION
test_log_pred, test_tpm_pred = predict_with_rc(model, test_seqs, log_mean, log_std)

# Ground truth (denormalized)
true_log = test_log
true_tpm = test_tpm

mse_log = np.mean((test_log_pred - true_log)**2)
mse_tpm = np.mean((test_tpm_pred - true_tpm)**2)

print("\n=== FINAL TEST PERFORMANCE (WITH RC AVERAGING) ===")
print("log-MSE:", mse_log)
print("TPM-MSE:", mse_tpm)
print("log RMSE:", np.sqrt(mse_log))
print("TPM RMSE:", np.sqrt(mse_tpm))


# ============================================================
# 9. WGAN scoring helper
# ============================================================
def evaluate_basenji_wgan(model, seq_list, log_mean, log_std):
    _, tpm_pred = predict_with_rc(model, seq_list, log_mean, log_std)
    return tpm_pred


Dataset size: 19942
Train/Val/Test: 16152 1795 1995

Using device: cuda
Epoch 01 | train=0.6592 | val=0.6144 | time=64.1s
Epoch 02 | train=0.6296 | val=0.6242 | time=65.2s
Epoch 03 | train=0.6235 | val=0.6473 | time=65.5s
Epoch 04 | train=0.6197 | val=0.6729 | time=64.8s
Epoch 05 | train=0.6135 | val=0.6241 | time=66.7s
Epoch 06 | train=0.6129 | val=0.8347 | time=66.1s
Epoch 07 | train=0.6075 | val=0.9345 | time=65.0s
Epoch 08 | train=0.6050 | val=0.6638 | time=64.7s
Epoch 09 | train=0.6002 | val=0.6259 | time=64.4s
Early stopping.

=== FINAL TEST PERFORMANCE (WITH RC AVERAGING) ===
log-MSE: 0.21510392414278975
TPM-MSE: 3.8689097758629365
log RMSE: 0.4637929755211799
TPM RMSE: 1.9669544417354807


In [57]:
# ============================================================
# SAVE MODEL + NORMALIZATION STATS
# ============================================================
save_path = "basenji_promoter_model.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "log_mean": log_mean,
    "log_std": log_std,
}, save_path)

print("Model saved to:", save_path)


Model saved to: basenji_promoter_model.pt


In [59]:
import torch
import pandas as pd

# -----------------------------
# 1. LOAD BASENJI MODEL + STATS
# -----------------------------
checkpoint = torch.load("basenji_promoter_model.pt", map_location="cuda" if torch.cuda.is_available() else "cpu")

log_mean = checkpoint["log_mean"]
log_std = checkpoint["log_std"]

# Recreate model architecture
model = BasenjiDilatedCNN()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded.")


# -----------------------------
# 2. LOAD GENERATED SEQUENCES
# -----------------------------
generated_sequences = []
with open("wgan_promoters_10000.txt", "r") as f:
    for line in f:
        seq = line.strip()
        if len(seq) > 0:
            generated_sequences.append(seq)

print("Loaded sequences:", len(generated_sequences))


# -----------------------------
# 3. SCORE USING RC AVERAGING
# -----------------------------
pred_log, pred_tpm = predict_with_rc(model, generated_sequences, log_mean, log_std)

# pred_log → predicted log(TPM+1)
# pred_tpm → predicted TPM


# -----------------------------
# 4. SORT BY TPM DESCENDING
# -----------------------------
df = pd.DataFrame({
    "sequence": generated_sequences,
    "pred_log_TPMp1": pred_log,
    "pred_TPM": pred_tpm
})

df_sorted = df.sort_values(by="pred_TPM", ascending=False)
df_sorted.head()


# -----------------------------
# 5. SAVE RESULTS
# -----------------------------
output_path = "wgan_10000_scored_sorted.csv"
df_sorted.to_csv(output_path, index=False)

print("Saved sorted results to:", output_path)


Model loaded.
Loaded sequences: 10000
Saved sorted results to: wgan_10000_scored_sorted.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance

# -------------------------------------------------------------
# Load data
# -------------------------------------------------------------
wgan_df = pd.read_csv("wgan_10000_scored_sorted.csv")
real_df = pd.read_csv("promoter_seq_TPM.csv")  # change name if needed

real_tpm = real_df["mean_TPM"].values
wgan_tpm = wgan_df["pred_TPM"].values

print("Loaded:", len(wgan_tpm), "WGAN seqs")
print("Loaded:", len(real_tpm), "Real promoters")

# -------------------------------------------------------------
# 1. Histograms (matplotlib only)
# -------------------------------------------------------------
plt.figure(figsize=(10,6))
plt.hist(real_tpm, bins=50, alpha=0.6, density=True, label="Real TPM", color="blue")
plt.hist(wgan_tpm, bins=50, alpha=0.6, density=True, label="WGAN TPM", color="red")
plt.title("Histogram: Real vs WGAN TPM")
plt.xlabel("TPM")
plt.ylabel("Density")
plt.legend()
plt.show()

# -------------------------------------------------------------
# 2. CDF Comparison (no seaborn)
# -------------------------------------------------------------
real_sorted = np.sort(real_tpm)
wgan_sorted = np.sort(wgan_tpm)

real_cdf = np.linspace(0, 1, len(real_sorted))
wgan_cdf = np.linspace(0, 1, len(wgan_sorted))

plt.figure(figsize=(10,6))
plt.plot(real_sorted, real_cdf, label="Real CDF", linewidth=2)
plt.plot(wgan_sorted, wgan_cdf, label="WGAN CDF", linewidth=2)
plt.xlabel("TPM")
plt.ylabel("CDF")
plt.title("CDF: Real vs WGAN TPM")
plt.legend()
plt.show()

# -------------------------------------------------------------
# 3. Boxplot (matplotlib only)
# -------------------------------------------------------------
plt.figure(figsize=(7,5))
plt.boxplot([real_tpm, wgan_tpm], labels=["Real", "WGAN"])
plt.title("Boxplot: Real vs WGAN TPM")
plt.ylabel("TPM")
plt.show()

# -------------------------------------------------------------
# 4. Top 50 WGAN Promoters scatter
# -------------------------------------------------------------
top50 = wgan_df.head(50)

plt.figure(figsize=(12,5))
plt.scatter(range(50), top50["pred_TPM"], color="red")
plt.title("Top 50 Highest-Scoring WGAN Promoters")
plt.xlabel("Rank")
plt.ylabel("TPM")
plt.grid(alpha=0.3)
plt.show()

# -------------------------------------------------------------
# 5. Statistics (still using scipy for accuracy)
# -------------------------------------------------------------
ks_stat, ks_p = ks_2samp(real_tpm, wgan_tpm)
wd = wasserstein_distance(real_tpm, wgan_tpm)

print("\n=== Distribution Statistics ===")
print(f"KS statistic: {ks_stat:.4f}")
print(f"KS p-value:   {ks_p:.4e}")
print(f"Wasserstein distance: {wd:.4f}")
print(f"Real mean TPM: {np.mean(real_tpm):.3f}")
print(f"WGAN mean TPM: {np.mean(wgan_tpm):.3f}")
print(f"Real TPM std:  {np.std(real_tpm):.3f}")
print(f"WGAN TPM std:  {np.std(wgan_tpm):.3f}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "d:\AnaConda\envs\genomics\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "d:\AnaConda\envs\genomics\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Akshat\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\Akshat\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", line 976, in launch_instance
    app.start()
  Fi


Original exception was:
AttributeError: _ARRAY_API not found


ImportError: numpy.core.multiarray failed to import

In [63]:
!pip uninstall numpy -y
!pip install "numpy<2"
!pip install --force-reinstall seaborn matplotlib scipy


Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.6.0 requires Pillow<10.0, but you have pillow 11.2.1 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.1 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)


  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\users\\akshat\\appdata\\roaming\\python\\python310\\site-packages\\kiwisolver.cp310-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



   ---------------------------------------- 8.1/8.1 MB 50.4 MB/s  0:00:00
   ---------------------------------------- 41.3/41.3 MB 36.0 MB/s  0:00:01
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
   ---------------------------------------- 1.5/1.5 MB 85.6 MB/s  0:00:00
   ---------------------------------------- 11.3/11.3 MB 78.6 MB/s  0:00:00
   ---------------------------------------- 7.0/7.0 MB 61.8 MB/s  0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2021.3
    Uninstalling pytz-2021.3:
      Successfully uninstalled pytz-2021.3
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2024.2
    Uninstalling tzdata-2024.2:
      Successfully uninstalled tzdata-2024.2
  Attempting uninstall: six
    Found existing installation: six 1.16.0
    Uninstalling six-1.16.0:
      Successfully uninstalled six-1.16.0
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 3.0.7
    Uninstalling pyparsing-3.0.7:
   